# Brain2Qwerty — Combined Pipeline (V1+V3 preprocessing, V3 architecture) on Kaggle

End-to-end sentence decoding from MEG with the V3 hybrid **Mamba-2/attention** encoder,
the **combined V1+V3 preprocessing**, the V1 TF-IDF paraphrase-cluster split, and optional
**subject subset selection**.

**Setup:**
1. Enable a GPU accelerator (Settings → Accelerator → GPU T4/P100).
2. Add your HF token as a Kaggle Secret named `HF_TOKEN` (Add-ons → Secrets) — the
   [SpanishBCBL dataset](https://huggingface.co/datasets/bcbl190626/SpanishBCBL) is gated.
3. Attach this project as a Kaggle Dataset (upload the `brain2qwerty_colab/` folder),
   or clone your fork in the cells below.

In [ ]:
# 1. GPU check
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 2. Install dependencies (torch is preinstalled on Kaggle images)
!pip install -q neuralset==0.2.2 neuraltrain==0.2.2 neuralfetch==0.2.2 exca==0.5.22 submitit==1.5.3 \
    lightning==2.5.2 torchmetrics==1.7.3 x-transformers==2.4.9 transformers==4.52.4 peft==0.18.1 \
    numpy==2.2.6 pandas==2.2.3 scipy==1.14.1 scikit-learn==1.8.0 pydantic==2.12.5 \
    mne==1.11.0 dtw-python==1.7.4 edit_distance==1.0.7 Levenshtein==0.27.1 g2p_en==2.1.0 \
    regex tqdm pyyaml huggingface_hub

In [ ]:
# 3. Locate the code (attached Kaggle dataset containing brain2qwerty_colab/)
import sys, os, glob

candidates = glob.glob('/kaggle/input/*') + ['/kaggle/working']
PROJECT_ROOT = next(
    (c for c in candidates if os.path.isdir(os.path.join(c, 'brain2qwerty_colab'))),
    None,
)
# Fallback: clone your fork
if PROJECT_ROOT is None:
    # !git clone https://github.com/<you>/brain2qwerty.git /kaggle/working/brain2qwerty
    # PROJECT_ROOT = '/kaggle/working/brain2qwerty'
    raise RuntimeError('brain2qwerty_colab/ not found under /kaggle/input — attach it as a dataset')
sys.path.insert(0, PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# 4. Paths + HF auth via Kaggle Secrets
import os
os.environ['BRAIN2QWERTY_STUDIES'] = '/kaggle/working/studies'
os.environ['BRAIN2QWERTY_CACHE']   = '/kaggle/working/cache'
os.environ['BRAIN2QWERTY_RESULTS'] = '/kaggle/working/results'

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

In [ ]:
# 4b. OPTIONAL: use a pre-warmed cache from your cluster instead of reprocessing
# 1) On the cluster:  python -m brain2qwerty_colab.main cache --subjects S1 S2
# 2) tar the $BRAIN2QWERTY_CACHE folder and upload it as a Kaggle dataset
# 3) Attach it here and point BRAIN2QWERTY_CACHE at the extracted copy:
import glob, os, tarfile
archives = glob.glob('/kaggle/input/*/*.tar.gz')
if archives:
    with tarfile.open(archives[0]) as tar:
        tar.extractall('/kaggle/working')
    extracted = glob.glob('/kaggle/working/*cache*')
    if extracted:
        os.environ['BRAIN2QWERTY_CACHE'] = extracted[0]
        print('using pre-warmed cache:', extracted[0])
# NOTE: cache keys are config-based — train with the SAME subjects/code version
# used to warm the cache, or lookups will miss and recompute (needs raw data).

In [ ]:
# 5. Choose subjects and preset (Kaggle sessions are ~9-12h: keep the subset small)
SUBJECTS = ['S1', 'S2']                 # None = all 19 participants
TIMELINE_QUERY = "subject in ['S1','S2']"  # skip loading other recordings entirely
SMALL_ENCODER = True                     # 512-dim encoder fits P100/T4 comfortably

from brain2qwerty_colab import studies  # registers the Pinet2024Meg study
from brain2qwerty_colab.config.xp_config import colab_config, debug_config

cfg = colab_config(subjects=SUBJECTS, timeline_query=TIMELINE_QUERY, small=SMALL_ENCODER)
print('subjects:', SUBJECTS or 'all 19', '| small encoder:', SMALL_ENCODER)

In [ ]:
# 6. Sanity-check the data path on one timeline first (downloads + preprocesses)
from brain2qwerty_colab.main import Experiment
Experiment(**debug_config(subjects=SUBJECTS)).data.build()
print('debug data path OK')

In [ ]:
# 7. Train (checkpoints land in /kaggle/working/results — downloadable from the Output tab)
xp = Experiment(**cfg)
xp.run()

In [ ]:
# 8. Evaluate on the test split + per-subject metrics
import os
ckpt = os.path.join(os.environ['BRAIN2QWERTY_RESULTS'], 'best_llm.ckpt')
eval_cfg = dict(cfg)
eval_cfg['eval_only'] = True
eval_cfg['ckpt_path'] = ckpt
Experiment(**eval_cfg).run()

from brain2qwerty_colab.scripts.extract_predictions import main as summarize
summarize(['--input', os.environ['BRAIN2QWERTY_RESULTS'], '--split', 'test'])

## Notes

- Everything lives under `/kaggle/working` — remember to download `results/` (checkpoints +
  `predictions_test.csv`) from the Output tab, or re-attach it as a dataset for the next session.
- The raw MEG dataset is large; `TIMELINE_QUERY` keeps only the selected subjects' recordings.
- If the session times out mid-training, rerun with `cfg['resume_ckpt'] = '.../last.ckpt'`.